# 03 · 合并与多级索引  Merge & MultiIndex

> **能力标签**：SH4（数据处理与分析）

## 目标 Objectives

完成本课后，你应该能够：

1. 用 `merge` 执行四种连接（`inner` / `left` / `right` / `outer`），理解各自保留哪些行。
2. 用 `concat` 沿行或列方向拼接多个 DataFrame。
3. 理解 **MultiIndex**（多级索引）的创建与基本操作。
4. 实现 `join_orders(customers, orders)` —— W3-merge 题包核心函数。

> 对应能力：**SH4**


## 概念讲解 Concepts

### `merge` 的四种 `how`

SQL 中的 JOIN 对应 pandas `merge` 的 `how` 参数：

| `how=` | 保留哪些行 | SQL 等价 |
|--------|-----------|----------|
| `"inner"`（默认）| 两表都有匹配的行 | INNER JOIN |
| `"left"` | 左表所有行 + 右表匹配行 | LEFT JOIN |
| `"right"` | 右表所有行 + 左表匹配行 | RIGHT JOIN |
| `"outer"` | 两表所有行（无匹配则 NaN）| FULL OUTER JOIN |

```python
customers.merge(orders, on="customer_id", how="inner")
```

---

### `concat`

把多个 DataFrame 堆叠（行方向 `axis=0`）或并排（列方向 `axis=1`）：

```python
pd.concat([df1, df2], axis=0, ignore_index=True)  # 纵向拼接
pd.concat([df_a, df_b], axis=1)                   # 横向拼接
```

---

### MultiIndex（多级索引）

当分组聚合后用 `as_index=True`，或显式 `set_index`，结果可能有多层 Index：

```python
midx = pd.MultiIndex.from_tuples([("A", 1), ("A", 2), ("B", 1)])
df_mi = pd.DataFrame({"val": [10, 20, 30]}, index=midx)
# 用 .loc["A"] 取第一层，.loc[("A", 1)] 取精确位置
```

---

### `join_orders` 函数签名（题包核心）

```python
def join_orders(customers, orders):
    return customers.merge(orders, on="customer_id", how="inner")
```


## 示例 Worked Example

In [ ]:
import pandas as pd
import numpy as np

# 构造示例表
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name':        ['Alice', 'Bob', 'Carol', 'Dave'],
    'city':        ['Beijing', 'Shanghai', 'Guangzhou', 'Shenzhen'],
})
orders = pd.DataFrame({
    'order_id':   [101, 102, 103, 104, 105],
    'customer_id': [1, 2, 2, 5, 3],   # 5 不在 customers 中，4 没有订单
    'amount':     [250.0, 130.0, 80.0, 400.0, 60.0],
})

print('=== customers ===')
print(customers)
print('\n=== orders ===')
print(orders)

# 1. inner join（只保留双方都有的）
inner = customers.merge(orders, on='customer_id', how='inner')
print('\n=== inner merge ===')
print(inner)

# 2. left join（保留所有 customer）
left = customers.merge(orders, on='customer_id', how='left')
print('\n=== left merge ===')
print(left)

# 3. outer join
outer = customers.merge(orders, on='customer_id', how='outer')
print('\n=== outer merge ===')
print(outer)

# 4. concat
extra_customers = pd.DataFrame({
    'customer_id': [5, 6], 'name': ['Eve', 'Frank'], 'city': ['Chengdu', 'Wuhan']
})
all_cust = pd.concat([customers, extra_customers], ignore_index=True)
print('\n=== concat customers ===')
print(all_cust)

# 5. MultiIndex 示例
print('\n=== MultiIndex ===')
grouped = orders.groupby(['customer_id', 'order_id'])['amount'].sum()
print(grouped)
print('Level names:', grouped.index.names)
print('loc customer_id=2:', grouped.loc[2])


## 动手 Exercise

在下面的 cell 中：

1. 创建两个 DataFrame：`products`（product_id, name, category）和 `sales`（sale_id, product_id, qty）。
2. 用 `inner merge` 把它们合并，只保留两表都有的 product_id。
3. 用 `left merge`，确认所有产品都出现（无销售的产品 qty 为 NaN）。
4. 用 `concat` 把两批销售记录纵向合并并重置索引。


In [ ]:
import pandas as pd

products = pd.DataFrame({
    'product_id': [1, 2, 3, 4],
    'name':       ['Keyboard', 'Mouse', 'Monitor', 'Headset'],
    'category':   ['Peripheral', 'Peripheral', 'Display', 'Audio'],
})
sales = pd.DataFrame({
    'sale_id':    [1001, 1002, 1003],
    'product_id': [1, 2, 1],
    'qty':        [3, 5, 2],
})

# 1. inner merge
inner = products.merge(sales, on='product_id', how='inner')
print('=== inner ===')
print(inner)

# 2. left merge
left = products.merge(sales, on='product_id', how='left')
print('\n=== left ===')
print(left)

# 3. concat 销售记录
more_sales = pd.DataFrame({
    'sale_id':    [1004, 1005],
    'product_id': [3, 4],
    'qty':        [1, 7],
})
all_sales = pd.concat([sales, more_sales], ignore_index=True)
print('\n=== all_sales ===')
print(all_sales)


## 延伸阅读 Further Reading

1. **pandas 官方文档 — merge**: <https://pandas.pydata.org/docs/user_guide/merging.html>
2. **pandas 官方文档 — MultiIndex**: <https://pandas.pydata.org/docs/user_guide/advanced.html>
3. **《Python for Data Analysis》第 8 章** — Wes McKinney
4. **SQL JOIN 可视化**: <https://joins.spathon.com/>


## 关联练习 Related Assignments

本课对应 **W3-merge** 题包，核心函数为 `join_orders`：

```bash
python check.py w3-merge
```

> 能力标签：**SH4** · threshold ≥ 0.7
